In [22]:
#Import the libraries 
import pandas as pd
import numpy as np
import seaborn as sns
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [26]:
# MELBOURNE HOUSING DATA ANALYSIS + XGBOOST

# 1. IMPORT LIBRARIES
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# 2. FIND AND LOAD DATA
filename = "melb_data_clean.csv"
folders = [
    os.getcwd(),
    os.path.expanduser("~/Downloads"),
    os.path.expanduser("~/Desktop"),
    os.path.expanduser("~/Documents")
]

file_path = next(
    (os.path.join(folder, filename) for folder in folders
     if os.path.exists(os.path.join(folder, filename))),
    None
)

if file_path is None:
    for root, dirs, files in os.walk(os.path.expanduser("~")):
        dirs[:] = [d for d in dirs if d not in ["Library", ".cache", ".local", "node_modules", ".git"]]
        if filename in files:
            file_path = os.path.join(root, filename)
            break

if file_path is None:
    raise FileNotFoundError(f"Could not find '{filename}'.")

print("CSV found:", file_path)

melb = pd.read_csv(file_path, parse_dates=["Date"])
print("\nDataset shape:", melb.shape)
print(melb.head())

# 3. BASIC DATA INFORMATION
print("\n--- Dataset Information ---")
print("Shape:", melb.shape)
print("\nColumns:", melb.columns.tolist())
print("\nData Types:")
print(melb.dtypes)
print("\nMissing Values:")
print(melb.isnull().sum())

# 4. DESCRIPTIVE STATISTICS
numeric_columns = [
    "Price", "Rooms", "Distance", "Bathroom", "Car",
    "Landsize", "BuildingArea", "YearBuilt"
]

numeric_columns = [col for col in numeric_columns if col in melb.columns]

print("\n--- Mean ---")
print(melb[numeric_columns].mean())

print("\n--- Median ---")
print(melb[numeric_columns].median())

print("\n--- Mode ---")
print(melb[numeric_columns].mode().iloc[0])

# 5. PRICE DISTRIBUTION
plt.figure(figsize=(10, 6))
melb["Price"].hist(bins=50)
plt.title("Melbourne House Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("melb_price_hist.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 6. PRICE BY PROPERTY TYPE
if "Type" in melb.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=melb, x="Type", y="Price")
    plt.title("Melbourne House Price by Property Type")
    plt.xlabel("Property Type")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.savefig("melb_price_by_type.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

# 7. PRICE VS DISTANCE
if "Distance" in melb.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=melb, x="Distance", y="Price", alpha=0.3)
    plt.title("House Price vs Distance from CBD")
    plt.xlabel("Distance from CBD")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.savefig("melb_price_vs_distance.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

# 8. CORRELATION
corr = melb[numeric_columns].corr()
print("\n--- Correlation with Price ---")
print(corr["Price"].sort_values(ascending=False))

# 9. CORRELATION HEATMAP
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Melbourne Housing Correlation Matrix")
plt.tight_layout()
plt.savefig("melb_correlation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 10. PREPARE DATA FOR XGBOOST
features = [
    "Rooms", "Distance", "Bathroom", "Car", "Landsize",
    "BuildingArea", "YearBuilt", "Latitude", "Longitude"
]

features = [col for col in features if col in melb.columns]
categorical = [col for col in ["Type", "Regionname"] if col in melb.columns]

X = melb[features].copy()

if categorical:
    encoded = pd.get_dummies(melb[categorical], drop_first=True)
    X = pd.concat([X, encoded], axis=1)

y = melb["Price"]

# 11. CLEAN DATA
X = X.replace([np.inf, -np.inf], np.nan)

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

for column in X.columns:
    if X[column].dtype in ["float64", "int64"]:
        X[column] = X[column].fillna(X[column].median())
    elif X[column].dtype == "bool":
        X[column] = X[column].astype(int)

print("\nMissing values after cleaning:", X.isnull().sum().sum())

# 12. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# 13. XGBOOST MODEL
model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror"
)

print("\nTraining XGBoost model...")
model.fit(X_train, y_train)
print("Training complete!")

# 14. PREDICTIONS
pred = model.predict(X_test)

# 15. MODEL PERFORMANCE
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print("\n--- Model Performance ---")
print(f"R² Score: {r2:.4f}")
print(f"MAE: ${mae:,.2f}")

# 16. FEATURE IMPORTANCE
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\n--- Top 10 Important Features ---")
print(importance.head(10))

# 17. FEATURE IMPORTANCE PLOT
plt.figure(figsize=(10, 7))
importance.head(10).sort_values().plot.barh()
plt.title("Top 10 Melbourne Housing Price Predictors")
plt.xlabel("Feature Importance")
plt.tight_layout()
plt.savefig("melb_importance.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 18. ACTUAL VS PREDICTED PRICE
plt.figure(figsize=(10, 6))
plt.scatter(y_test, pred, alpha=0.4)

min_price = min(y_test.min(), pred.min())
max_price = max(y_test.max(), pred.max())

plt.plot([min_price, max_price], [min_price, max_price])
plt.title("Actual vs Predicted Melbourne House Prices")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.tight_layout()
plt.savefig("melb_actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 19. SAVE PREDICTIONS
results = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": pred
})

results["Absolute_Error"] = (
    results["Actual_Price"] - results["Predicted_Price"]
).abs()

results.to_csv("melb_predictions.csv", index=False)

# 20. FINAL SUMMARY
print("\nANALYSIS COMPLETE")
print(f"Dataset size: {melb.shape[0]:,} rows × {melb.shape[1]} columns")
print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Error: ${mae:,.2f}")

print("\nFiles created:")
print("1. melb_price_hist.png")
print("2. melb_price_by_type.png")
print("3. melb_price_vs_distance.png")
print("4. melb_correlation.png")
print("5. melb_importance.png")
print("6. melb_actual_vs_predicted.png")
print("7. melb_predictions.csv")

CSV found: /Users/jahaankachwala/anaconda_projects/47ac4618-a28f-45c2-905a-fd494867325a/melb_data_clean.csv

Dataset shape: (13579, 21)
           Suburb               Address  Rooms         Type      Price  \
0    Surrey Hills    999A Riversdale Rd      3        House  1205000.0   
1    Surrey Hills      1/10 Florence Rd      2  Unit/Duplex   813000.0   
2     Mont Albert  50/781 Whitehorse Rd      2  Unit/Duplex   750000.0   
3          Ormond         7/30 Walsh St      1  Unit/Duplex   250000.0   
4  Port Melbourne        404/101 Bay St      2  Unit/Duplex   550000.0   

       Method    SellerG       Date  Distance  Postcode  ...  Bathroom  Car  \
0        Sold  Fletchers 2016-01-28      11.2      3127  ...         1  2.0   
1        Sold  Fletchers 2016-01-28      11.2      3127  ...         1  2.0   
2        Sold         RW 2016-02-04      11.8      3127  ...         2  2.0   
3  Vendor bid        Ray 2016-02-04      11.8      3204  ...         1  1.0   
4        Sold     Buxton

/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/3534313275.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/3534313275.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/3534313275.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- Correlation with Price ---
Price           1.000000
BuildingArea    0.547266
Rooms           0.496642
Bathroom        0.467079
Car             0.239000
Landsize        0.007042
Distance       -0.162541
YearBuilt      -0.331784
Name: Price, dtype: float64


/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/3534313275.py:116: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Missing values after cleaning: 0

Training rows: 10863
Testing rows: 2716

Training XGBoost model...
Training complete!

--- Model Performance ---
R² Score: 0.8289
MAE: $158,888.67

--- Top 10 Important Features ---
Regionname_Southern Metropolitan         0.500073
Type_Unit/Duplex                         0.142372
Rooms                                    0.084198
Bathroom                                 0.069089
Type_Townhouse                           0.037903
Distance                                 0.033097
BuildingArea                             0.023657
Landsize                                 0.019871
Regionname_South-Eastern Metropolitan    0.018185
Longitude                                0.016721
dtype: float32

ANALYSIS COMPLETE
Dataset size: 13,579 rows × 21 columns
R² Score: 0.8289
Mean Absolute Error: $158,888.67

Files created:
1. melb_price_hist.png
2. melb_price_by_type.png
3. melb_price_vs_distance.png
4. melb_correlation.png
5. melb_importance.png
6. melb_actual_vs_

/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/3534313275.py:199: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/3534313275.py:215: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
# NORTHERN TERRITORY REAL ESTATE ANALYSIS + XGBOOST

# 1. IMPORT LIBRARIES
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# 2. FIND AND LOAD DATA
filename = "RealEstateAU_NT_clean.csv"

folders = [
    os.getcwd(),
    os.path.expanduser("~/Downloads"),
    os.path.expanduser("~/Desktop"),
    os.path.expanduser("~/Documents")
]

file_path = next(
    (os.path.join(folder, filename) for folder in folders
     if os.path.exists(os.path.join(folder, filename))),
    None
)

if file_path is None:
    for root, dirs, files in os.walk(os.path.expanduser("~")):
        dirs[:] = [
            d for d in dirs
            if d not in ["Library", ".cache", ".local", "node_modules", ".git"]
        ]
        if filename in files:
            file_path = os.path.join(root, filename)
            break

if file_path is None:
    raise FileNotFoundError(f"Could not find '{filename}'.")

print("CSV found:", file_path)

nt = pd.read_csv(file_path)

print("\nDataset shape:", nt.shape)
print(nt.head())

# 3. BASIC DATA INFORMATION
print("\n--- Dataset Information ---")
print("Shape:", nt.shape)
print("\nColumns:", nt.columns.tolist())
print("\nData Types:")
print(nt.dtypes)
print("\nMissing Values:")
print(nt.isnull().sum())

# 4. DESCRIPTIVE STATISTICS
numeric_columns = [
    "price_num",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "building_size_m2",
    "land_size_m2"
]

numeric_columns = [
    col for col in numeric_columns
    if col in nt.columns
]

print("\n--- Mean ---")
print(nt[numeric_columns].mean())

print("\n--- Median ---")
print(nt[numeric_columns].median())

print("\n--- Mode ---")
print(nt[numeric_columns].mode().iloc[0])

# 5. PRICE DISTRIBUTION
plt.figure(figsize=(10, 6))
nt["price_num"].hist(bins=40)
plt.title("Northern Territory Property Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("nt_price_hist.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 6. PRICE BY PROPERTY TYPE
if "property_type" in nt.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=nt,
        x="property_type",
        y="price_num"
    )
    plt.xticks(rotation=45)
    plt.title("Northern Territory Price by Property Type")
    plt.xlabel("Property Type")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.savefig("nt_price_by_type.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

# 7. CORRELATION
corr = nt[numeric_columns].corr()

print("\n--- Correlation with Price ---")
print(corr["price_num"].sort_values(ascending=False))

# 8. CORRELATION HEATMAP
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Northern Territory Real Estate Correlation")
plt.tight_layout()
plt.savefig("nt_correlation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 9. PREPARE DATA FOR XGBOOST
nt = nt.dropna(subset=["price_num"])

features = [
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "building_size_m2",
    "land_size_m2",
    "postcode"
]

features = [
    col for col in features
    if col in nt.columns
]

X = nt[features].copy()

# 10. PROPERTY TYPE ENCODING
if "property_type" in nt.columns:
    encoded = pd.get_dummies(
        nt["property_type"],
        drop_first=True
    )
    X = pd.concat([X, encoded], axis=1)

y = nt["price_num"]

# 11. CLEAN DATA
X = X.replace([np.inf, -np.inf], np.nan)

if "postcode" in X.columns:
    X["postcode"] = pd.to_numeric(
        X["postcode"],
        errors="coerce"
    )

for column in X.columns:
    if pd.api.types.is_numeric_dtype(X[column]):
        X[column] = X[column].fillna(X[column].median())
    elif X[column].dtype == "bool":
        X[column] = X[column].astype(int)

print("\nMissing values after cleaning:", X.isnull().sum().sum())

# 12. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# 13. XGBOOST MODEL
model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    objective="reg:squarederror"
)

print("\nTraining XGBoost model...")
model.fit(X_train, y_train)
print("Training complete!")

# 14. PREDICTIONS
pred = model.predict(X_test)

# 15. MODEL PERFORMANCE
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print("\n--- Model Performance ---")
print(f"R² Score: {r2:.4f}")
print(f"MAE: ${mae:,.2f}")

# 16. FEATURE IMPORTANCE
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\n--- Top 10 Important Features ---")
print(importance.head(10))

# 17. FEATURE IMPORTANCE PLOT
plt.figure(figsize=(10, 7))
importance.head(10).sort_values().plot.barh()
plt.title("Top 10 Northern Territory Property Price Predictors")
plt.xlabel("Feature Importance")
plt.tight_layout()
plt.savefig("nt_importance.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 18. ACTUAL VS PREDICTED PRICE
plt.figure(figsize=(10, 6))
plt.scatter(y_test, pred, alpha=0.4)

min_price = min(y_test.min(), pred.min())
max_price = max(y_test.max(), pred.max())

plt.plot(
    [min_price, max_price],
    [min_price, max_price]
)

plt.title("Actual vs Predicted Northern Territory Property Prices")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.tight_layout()
plt.savefig("nt_actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 19. SAVE PREDICTIONS
results = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": pred
})

results["Absolute_Error"] = (
    results["Actual_Price"] -
    results["Predicted_Price"]
).abs()

results.to_csv("nt_predictions.csv", index=False)

# 20. FINAL SUMMARY
print("\nNORTHERN TERRITORY ANALYSIS COMPLETE")
print(f"Dataset size: {nt.shape[0]:,} rows × {nt.shape[1]} columns")
print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Error: ${mae:,.2f}")

print("\nFiles created:")
print("1. nt_price_hist.png")
print("2. nt_price_by_type.png")
print("3. nt_correlation.png")
print("4. nt_importance.png")
print("5. nt_actual_vs_predicted.png")
print("6. nt_predictions.csv")

CSV found: /Users/jahaankachwala/anaconda_projects/47ac4618-a28f-45c2-905a-fd494867325a/RealEstateAU_NT_clean.csv

Dataset shape: (999, 24)
       TID  location_number property_type             price_raw  price_num  \
0  1350988        139468611         House              $435,000   435000.0   
1  1350989        139463755     Apartment  Offers Over $320,000   320000.0   
2  1350990        139462495          Unit              $310,000   310000.0   
3  1350991        139451679         House              $259,000   259000.0   
4  1350992        139433803          Unit              $439,000   439000.0   

   price_min  price_max    price_type listing_status listed_date  ...  \
0   435000.0   435000.0  Fixed/asking         Active  2022-05-27  ...   
1   320000.0   320000.0   Offers over         Active  2022-05-27  ...   
2   310000.0   310000.0  Fixed/asking         Active  2022-05-26  ...   
3   259000.0   259000.0  Fixed/asking         Active  2022-05-26  ...   
4   439000.0   439000.0  F

/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/1707550251.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/1707550251.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- Correlation with Price ---
price_num           1.000000
bedrooms            0.539061
bathrooms           0.516660
parking_spaces      0.381800
building_size_m2    0.125380
land_size_m2        0.087769
Name: price_num, dtype: float64

Missing values after cleaning: 0

Training rows: 621
Testing rows: 156

Training XGBoost model...


/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/1707550251.py:127: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Training complete!

--- Model Performance ---
R² Score: 0.2399
MAE: $106,124.86

--- Top 10 Important Features ---
bedrooms                0.321713
bathrooms               0.219557
Residential Land        0.087549
House                   0.073734
building_size_m2        0.060386
postcode                0.058451
land_size_m2            0.040933
parking_spaces          0.039391
Duplex/Semi-detached    0.021939
Townhouse               0.021882
dtype: float32

NORTHERN TERRITORY ANALYSIS COMPLETE
Dataset size: 777 rows × 24 columns
R² Score: 0.2399
Mean Absolute Error: $106,124.86

Files created:
1. nt_price_hist.png
2. nt_price_by_type.png
3. nt_correlation.png
4. nt_importance.png
5. nt_actual_vs_predicted.png
6. nt_predictions.csv


/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/1707550251.py:227: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_90649/1707550251.py:247: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
